# EXP-024 | Interaction Target Encoding

EXP-023(단일 TE)에 **Interaction TE** 추가: 두 컬럼을 조합한 범주의 임신 성공률 인코딩.

| 조합 | 임상 의미 |
|---|---|
| 나이 × 시술유형 | 나이대별 IVF/DI 성공률 차이 |
| 나이 × 특정시술유형 | 나이대별 세부 시술 성공률 |
| 나이 × 난자출처 | 기증란은 나이 영향 감소 (임상적 핵심 상호작용) |
| 나이 × 배란유도유형 | 나이별 최적 과배란 프로토콜 |
| 시술유형 × 총시술횟수 | 시술 종류별 반복 경험 효과 |
| 난자출처 × 시술유형 | 난자 출처와 시술 방식의 조합 |

| 항목 | 내용 |
|---|---|
| 단일 TE | EXP-023 동일 (9개, alpha=10) |
| Interaction TE | 6쌍, alpha=20 (소수 조합 안정화) |
| 베이스 피처 | EXP-021 (FE v2) |
| 파라미터 | LGB(EXP-019) / CAT(EXP-011) / XGB(EXP-012) |
| 기준선 | EXP-023 = 0.74063 |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, recall_score, precision_score, accuracy_score)
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 24
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비 (FE v2)

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    """EXP-003 피처 + EXP-021 v2 신규 4개"""
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']     = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']       = df[male_cols].sum(axis=1)
    df['여성_불임_합계']       = df[female_cols].sum(axis=1)
    df['부부_불임_합계']       = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']       = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']       = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']     = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']   = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    df['임신_성공률']   = df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)
    df['시술_실패_횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    egg_count = df['수집된 신선 난자 수'].fillna(-1)
    df['최적_난자수']    = ((egg_count >= 5) & (egg_count <= 15)).astype(int)
    df['나이_이식배아수'] = df['시술 당시 나이'] * df['이식된 배아 수'].fillna(0)
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 89)  /  X_test: (90067, 89)


## 2. TE 설정 (단일 TE + Interaction TE)

In [3]:
# ── 단일 TE (EXP-023 동일) ────────────────────────────────────────────────
_TE_CANDIDATES = [
    '시술 시기 코드', '시술 유형', '특정 시술 유형',
    '배란 유도 유형', '난자 출처', '정자 출처',
    '시술 당시 나이', '총 시술 횟수', '총 임신 횟수',
]
TE_COLS  = [c for c in _TE_CANDIDATES if c in X_train.columns]
TE_ALPHA = 10

# ── Interaction TE ────────────────────────────────────────────────────────
# alpha=20: 조합 범주는 셀당 샘플이 적으므로 더 강한 스무딩
_ITE_CANDIDATES = [
    ('시술 당시 나이', '시술 유형'),       # 나이 × 시술 종류
    ('시술 당시 나이', '특정 시술 유형'),   # 나이 × 세부 시술
    ('시술 당시 나이', '난자 출처'),        # 나이 × 난자 출처 (기증란은 나이 영향 감소)
    ('시술 당시 나이', '배란 유도 유형'),   # 나이 × 과배란 프로토콜
    ('시술 유형',     '총 시술 횟수'),      # 시술 종류 × 반복 경험
    ('난자 출처',     '시술 유형'),         # 난자 출처 × 시술 종류
]
ITE_PAIRS = [(c1, c2) for c1, c2 in _ITE_CANDIDATES
             if c1 in X_train.columns and c2 in X_train.columns]
ITE_ALPHA = 20

print(f'단일  TE: {len(TE_COLS)}개 컬럼  (alpha={TE_ALPHA})')
print(f'상호작용 TE: {len(ITE_PAIRS)}쌍  (alpha={ITE_ALPHA})')
print(f'추가 피처 수: {len(TE_COLS) + len(ITE_PAIRS)}개')
print(f'최종 피처 수 예상: {X_train.shape[1] + len(TE_COLS) + len(ITE_PAIRS)}')

단일  TE: 9개 컬럼  (alpha=10)
상호작용 TE: 6쌍  (alpha=20)
추가 피처 수: 15개
최종 피처 수 예상: 104


## 3. 학습 (단일 TE + Interaction TE)

각 fold에서 train으로 TE 통계 계산 → val/test에 적용.

In [4]:
LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824, num_leaves=266, max_depth=5,
    min_child_samples=166, feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091, lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442, min_gain_to_split=0.11418176854933762,
)
CAT_PARAMS = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', random_seed=SEED,
    thread_count=-1, verbose=False, early_stopping_rounds=100,
    learning_rate=0.018758723768855998, depth=6,
    l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483, colsample_bylevel=0.6936810336930781,
)
XGB_PARAMS = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist', random_state=SEED,
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647, max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595, colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928, reg_lambda=0.23932562420374562,
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb  = np.zeros(len(X_train)); test_lgb = np.zeros(len(X_test))
oof_cat  = np.zeros(len(X_train)); test_cat = np.zeros(len(X_test))
oof_xgb  = np.zeros(len(X_train)); test_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr  = X_train.iloc[tr_idx].copy()
    X_val = X_train.iloc[val_idx].copy()
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    X_te  = X_test.copy()
    global_mean = y_tr.mean()

    # ── 단일 Target Encoding ──────────────────────────────────────────
    for col in TE_COLS:
        te_col = f'{col}_te'
        grp    = y_tr.groupby(X_tr[col])
        cnt    = grp.count()
        mean_t = grp.mean()
        smoothed = (cnt * mean_t + TE_ALPHA * global_mean) / (cnt + TE_ALPHA)
        X_tr[te_col]  = X_tr[col].map(smoothed).fillna(global_mean)
        X_val[te_col] = X_val[col].map(smoothed).fillna(global_mean)
        X_te[te_col]  = X_te[col].map(smoothed).fillna(global_mean)

    # ── Interaction Target Encoding ───────────────────────────────────
    for col1, col2 in ITE_PAIRS:
        ite_col  = f'{col1}_{col2}_ite'
        key_tr   = X_tr[col1].astype(str)  + '_' + X_tr[col2].astype(str)
        key_val  = X_val[col1].astype(str) + '_' + X_val[col2].astype(str)
        key_te   = X_te[col1].astype(str)  + '_' + X_te[col2].astype(str)
        grp      = y_tr.groupby(key_tr)
        cnt      = grp.count()
        mean_t   = grp.mean()
        smoothed = (cnt * mean_t + ITE_ALPHA * global_mean) / (cnt + ITE_ALPHA)
        X_tr[ite_col]  = key_tr.map(smoothed).fillna(global_mean)
        X_val[ite_col] = key_val.map(smoothed).fillna(global_mean)
        X_te[ite_col]  = key_te.map(smoothed).fillna(global_mean)
    # ─────────────────────────────────────────────────────────────────

    # LGB
    ds_tr  = lgb.Dataset(X_tr, label=y_tr)
    ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    m = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx] = m.predict(X_val); test_lgb += m.predict(X_te) / N_FOLDS

    # CAT
    m = CatBoostClassifier(**CAT_PARAMS)
    m.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = m.predict_proba(X_val)[:, 1]; test_cat += m.predict_proba(X_te)[:, 1] / N_FOLDS

    # XGB
    m = xgb.XGBClassifier(**XGB_PARAMS)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m.predict_proba(X_val)[:, 1]; test_xgb += m.predict_proba(X_te)[:, 1] / N_FOLDS

    print(f'  Fold {fold}  LGB={roc_auc_score(y_val, oof_lgb[val_idx]):.5f}  '
          f'CAT={roc_auc_score(y_val, oof_cat[val_idx]):.5f}  '
          f'XGB={roc_auc_score(y_val, oof_xgb[val_idx]):.5f}')

auc_lgb = roc_auc_score(y_train, oof_lgb)
auc_cat = roc_auc_score(y_train, oof_cat)
auc_xgb = roc_auc_score(y_train, oof_xgb)
n_features_final = X_tr.shape[1]
print(f'\nOOF  LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}')
print(f'최종 피처 수: {n_features_final}')

  Fold 1  LGB=0.73819  CAT=0.73830  XGB=0.73833
  Fold 2  LGB=0.74314  CAT=0.74313  XGB=0.74301
  Fold 3  LGB=0.74046  CAT=0.74053  XGB=0.74036
  Fold 4  LGB=0.73863  CAT=0.73853  XGB=0.73893
  Fold 5  LGB=0.74136  CAT=0.74085  XGB=0.74156

OOF  LGB=0.74035  CAT=0.74026  XGB=0.74043
최종 피처 수: 104


## 4. 앙상블 & 결과 비교

In [5]:
oofs  = np.stack([oof_lgb, oof_cat, oof_xgb], axis=1)
tests = np.stack([test_lgb, test_cat, test_xgb], axis=1)
aucs  = np.array([auc_lgb, auc_cat, auc_xgb])

results = {}
results['Simple Average'] = (roc_auc_score(y_train, oofs.mean(axis=1)),
                              oofs.mean(axis=1), tests.mean(axis=1))

w_auc = aucs / aucs.sum()
results['AUC-weighted'] = (roc_auc_score(y_train, (oofs * w_auc).sum(axis=1)),
                            (oofs * w_auc).sum(axis=1), (tests * w_auc).sum(axis=1))

oof_ranks  = np.stack([rankdata(oofs[:, i]) for i in range(3)], axis=1)
test_ranks = np.stack([rankdata(tests[:, i]) for i in range(3)], axis=1)
results['Rank Average'] = (roc_auc_score(y_train, oof_ranks.mean(axis=1)),
                            oof_ranks.mean(axis=1), test_ranks.mean(axis=1))

def optuna_obj(trial):
    w = np.array([trial.suggest_float('w_lgb', 0, 1),
                  trial.suggest_float('w_cat', 0, 1),
                  trial.suggest_float('w_xgb', 0, 1)])
    w /= w.sum()
    return roc_auc_score(y_train, (oofs * w).sum(axis=1))

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(optuna_obj, n_trials=300, show_progress_bar=False)
best_w = np.array([study.best_params['w_lgb'], study.best_params['w_cat'],
                   study.best_params['w_xgb']])
best_w /= best_w.sum()
results['Optuna Weights'] = (roc_auc_score(y_train, (oofs * best_w).sum(axis=1)),
                              (oofs * best_w).sum(axis=1), (tests * best_w).sum(axis=1))

BASELINE_023 = 0.74063
BASELINE_021 = 0.74048
print('=' * 70)
print(f'  개별: LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}')
print(f'  EXP-023 기준선: {BASELINE_023}  /  EXP-021: {BASELINE_021}')
print('-' * 70)
best_method, best_auc, best_oof_pred, best_test = '', 0, None, None
for method, (auc, oof_pred, test_pred) in results.items():
    flag = ' ←' if auc > max(aucs) else ''
    print(f'  {method:20s}: {auc:.5f}  '
          f'({auc - BASELINE_023:+.5f} vs EXP-023){flag}')
    if auc > best_auc:
        best_auc, best_method, best_oof_pred, best_test = auc, method, oof_pred, test_pred
print('=' * 70)
print(f'\n최적: {best_method}  AUC={best_auc:.5f}')
print(f'Optuna 가중치  LGB={best_w[0]:.3f}  CAT={best_w[1]:.3f}  XGB={best_w[2]:.3f}')

  개별: LGB=0.74035  CAT=0.74026  XGB=0.74043
  EXP-023 기준선: 0.74063  /  EXP-021: 0.74048
----------------------------------------------------------------------
  Simple Average      : 0.74061  (-0.00002 vs EXP-023) ←
  AUC-weighted        : 0.74061  (-0.00002 vs EXP-023) ←
  Rank Average        : 0.74061  (-0.00002 vs EXP-023) ←
  Optuna Weights      : 0.74061  (-0.00002 vs EXP-023) ←

최적: Rank Average  AUC=0.74061
Optuna 가중치  LGB=0.259  CAT=0.352  XGB=0.389


## 5. Submission 저장 & 실험 기록

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sub['probability'] = best_test
auc_str   = f'{best_auc:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}')

저장: ../data/submissions/submission_exp024_조여진_07406.csv


In [7]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (best_oof_pred >= 0.5).astype(int)
ite_names  = [f'{c1}×{c2}' for c1, c2 in ITE_PAIRS]
params_str = (f'FE-v2 + TE({len(TE_COLS)}cols) + ITE({len(ITE_PAIRS)}pairs) | '
              f'LGB(EXP-019)+CAT(EXP-011)+XGB(EXP-012) | best={best_method}')
NOTES    = f'Interaction TE {len(ITE_PAIRS)}쌍 추가: {", ".join(ite_names)}'
INSIGHTS = (f'EXP-023 대비 {best_auc - BASELINE_023:+.5f} | '
            f'base LGB={auc_lgb:.5f} CAT={auc_cat:.5f} XGB={auc_xgb:.5f} | '
            f'피처 수 {n_features_final}')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'Ensemble(FE-v2+TE+ITE)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    best_auc, CV_STR, 'v2+TE+ITE', n_features_final,
    'is_unbalance / Balanced / scale_pos_weight',
    'N', None, 'notebooks/20_interaction_te_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-024 기록 완료 (row 22)
